# MCP Server & CLI Walkthrough

A practical guide to the two runtime surfaces of the SDK:

- **MCP server** (`odoo-mcp` / `python -m odoo_sdk.mcp`) — exposes Odoo operations as typed tools that a Claude agent can call directly.
- **CLI companion** (`python -m odoo_sdk.cli`) — a human-facing terminal tool for inspecting and managing active time-tracking sessions.

> **Note:** `%%bash` cells run in a subprocess — environment variables set in one cell do **not** persist to the next. Use the INI file approach below for credentials that all cells can read.

## Prerequisites

Install and sync the project.

In [ ]:
%%bash
cd /workspaces/devcontainer-features/libraries
uv sync

## Configure a Connection

Settings resolve in this order: explicit constructor args → environment variables → INI file.

The INI file is the most convenient option for a notebook because it persists across cells.
Create `.odoo_sdk.ini` in the project root (or `~/.config/odoo_sdk/config.ini` for a global default):

In [ ]:
%%bash
cat > /workspaces/devcontainer-features/libraries/.odoo_sdk.ini <<'EOF'
[odoo]
url      = https://your-instance.odoo.com
db       = your-database
username = you@example.com
password = your-password
EOF
echo "Written .odoo_sdk.ini"

For JSON-RPC2 with an API key, use this variant instead:

In [ ]:
%%bash
cat > /workspaces/devcontainer-features/libraries/.odoo_sdk.ini <<'EOF'
[odoo]
url       = https://your-instance.odoo.com
db        = your-database
api_key   = your-api-key
transport = json2
EOF
echo "Written .odoo_sdk.ini (json2)"

Alternatively, export environment variables at the top of the notebook using Python so they persist for every `%%bash` cell in this session:

In [ ]:
import os

os.environ["ODOO_URL"]      = "https://your-instance.odoo.com"
os.environ["ODOO_DB"]       = "your-database"
os.environ["ODOO_USERNAME"] = "you@example.com"
os.environ["ODOO_PASSWORD"] = "your-password"

---

## MCP Server

The MCP server exposes every registered command as a typed tool that Claude can call.
The default entry point includes all built-in commands.

### Starting the server

The server communicates over **stdio** and runs until killed — run it in a dedicated terminal, not inside this notebook.
The cell below is provided for reference; executing it in a notebook will block the kernel.

In [ ]:
%%bash
# Run in a terminal — this blocks until killed.
# odoo-mcp
#
# Equivalent module form:
# python -m odoo_sdk.mcp
echo "(start odoo-mcp in a separate terminal)"

### Register with Claude Code

Tell Claude Code where to find the server so it is available in every session.

In [ ]:
%%bash
claude mcp add odoo-sdk -- odoo-mcp

In [ ]:
%%bash
claude mcp list

### Built-in tools

Once the server is registered, paste these prompts directly into Claude Code to call each tool.

---

**Verify connectivity**
```
Use the get_uid tool to confirm I am authenticated to Odoo.
```

**List all Odoo models**
```
Use get_models to list every model on this Odoo instance.
```

**Browse your tasks**
```
Use task_list to show all tasks assigned to me.
```
```
Use task_list with project_name_query="Website" to show only tasks in the Website project.
```
```
Use task_list with stage="In Progress" and limit=5 to show my in-progress tasks.
```

**Search tasks with a raw domain filter**
```
Use get_tasks with domain=[["stage_id.name","=","Done"]] and limit=10.
```

**Fetch a single task by id**
```
Use get_todo with task_id=42 to fetch that specific task.
```

**Create a task**
```
Use create_task to create a task called "Review PR" in project 7 with description="Quick pass on the open PR".
```

**Start time-tracking a task** *(interactive — Claude will ask for disambiguation and confirmation)*
```
Use start_task with task_name_query="Review PR" and project_name_query="Website".
```

**Check what is currently being tracked**
```
Use task_status to see all active tracking sessions with elapsed time.
```

**Post a chatter note to a task**
```
Use task_note with task_id=42 and note="Investigated the login regression — traced to a cookie path mismatch.".
```

**Ask a stakeholder question** *(transitions session to AWAITING_ANSWERS)*
```
Use task_question with task_id=42 and question="Should the fallback redirect go to /web or /login?".
```

**Resume a paused session after getting answers**
```
Use resume_task with task_id=42.
```

**Stop tracking and finalize the timesheet** *(Claude presents the description for your review before writing to Odoo)*
```
Use stop_task with task_id=42 and description="Investigated cookie-path regression, identified fix, opened PR #17.".
```

---

## CLI Companion

The CLI is a human-readable terminal interface for the local session state database.
It does not replace the MCP tools — it is a sidecar for inspecting and emergency-managing sessions without going through an agent.

All subcommands check that the current shell is an Odoo devcontainer and exit with a clear error message if not.

### `list` — show active sessions (default)

In [ ]:
%%bash
python -m odoo_sdk.cli list

In [ ]:
%%bash
# list is the default subcommand
python -m odoo_sdk.cli

### `report` — formatted session table

In [ ]:
%%bash
# Active sessions only
python -m odoo_sdk.cli report

In [ ]:
%%bash
# Include stopped sessions
python -m odoo_sdk.cli report --all

### `stop` — force-stop one session

Use the `[ID]` from `list`. This finalizes elapsed time in Odoo and marks the session STOPPED.

In [ ]:
%%bash
# Replace 3 with the session ID shown by 'list'
python -m odoo_sdk.cli stop 3

### `stop-all` — force-stop every active session

In [ ]:
%%bash
python -m odoo_sdk.cli stop-all

### `normalize` — merge duplicate timesheet entries

If you started and stopped the same task multiple times in one day, Odoo ends up with multiple timesheet entries for that task and date.
`normalize` detects those duplicates and optionally merges them.

In [ ]:
%%bash
# Dry-run — shows what would be merged without touching Odoo
python -m odoo_sdk.cli normalize

In [ ]:
%%bash
# Apply — executes the merge in Odoo and updates local state
python -m odoo_sdk.cli normalize --apply

---

## Typical End-to-End Workflow

A complete session from picking up a task to logging the work.
Steps 1–6 are Claude prompts; steps 5 and 7 are CLI commands you can run in the cells below.

**1. See what's assigned to you**
```
Use task_list to show what's assigned to me this week.
```

**2. Start tracking a task**
```
Use start_task with task_name_query="Fix login" and project_name_query="Backend".
```

**3. When you need to ask a stakeholder something**
```
Use task_question with task_id=42 and question="Which environment should the fix target first — staging or production?".
```

**4. When the answer arrives, resume**
```
Use resume_task with task_id=42.
```

**5. Check elapsed time from the terminal**

In [ ]:
%%bash
python -m odoo_sdk.cli

**6. Stop the session — Claude generates the description, you confirm before it writes to Odoo**
```
Stop task 42. I fixed the cookie-path mismatch in the session middleware and added a regression test.
```

**7. At end-of-day, clean up any duplicate timesheet entries**

In [ ]:
%%bash
python -m odoo_sdk.cli normalize

In [ ]:
%%bash
# If duplicates were shown above, apply the merge:
python -m odoo_sdk.cli normalize --apply

---

## Adding Custom MCP Tools

Define a `Command` subclass, register it on a `Registry`, and start `OdooMCPServer` yourself.
The typed `execute` signature is introspected automatically — no schema boilerplate needed.

The cell below writes a ready-to-run server script:

In [ ]:
script = """\
from typing import Any, Dict, List
from odoo_sdk import OdooClient, Registry
from odoo_sdk.commands import Command
from odoo_sdk.mcp import OdooMCPServer


class SearchPartnersCommand(Command):
    _name = "search_partners"
    _description = "Search contacts whose name matches a fragment."

    def execute(self, name: str, limit: int = 20) -> List[Dict[str, Any]]:
        return (
            self._client["res.partner"]
            .search([("name", "ilike", name)], limit=limit)
            .read(["name", "email"])
        )


client = OdooClient()
registry = Registry(client)
registry.register("search_partners", SearchPartnersCommand)

# Optionally include the built-in tools too:
# from odoo_sdk.commands.builtin import register_builtins
# register_builtins(registry)

OdooMCPServer(registry, server_name="My Odoo MCP Server").run()
"""

with open("/tmp/my_odoo_server.py", "w") as f:
    f.write(script)

print("Written to /tmp/my_odoo_server.py")

Register the custom server with Claude Code and verify it appears:

In [ ]:
%%bash
claude mcp add my-odoo -- python /tmp/my_odoo_server.py

In [ ]:
%%bash
claude mcp list

Then use the new tool in Claude:

```
Use search_partners with name="Acme" to find all Acme contacts.
```